# Feature Engineering: WatchNext

This notebook demonstrates the process of transforming raw MovieLens data into features suitable for machine learning models. We cover user profiling, content-based item features (Genres & Tags), and the construction of the sparse interaction matrix.

## 1. Setup and Imports

We import the modular feature engineering functions from the `watchnext` package.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data_loader import load_raw_data
from src.features.build_features import (
    build_user_features,
    build_genre_features,
    build_tag_features,
    build_item_features,
    build_interaction_matrix
)

%matplotlib inline
sns.set_theme(style="whitegrid")

In [ ]:
dataset = load_raw_data()
print("Tables loaded:", list(dataset.keys()))

## 2. User Feature Engineering

We calculate user-level statistics, including average ratings, activity span, and a recency weight that discounts older ratings.

In [ ]:
user_features = build_user_features(dataset["ratings"])
print(f"User features shape: {user_features.shape}")
user_features.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(user_features["mean_rating"], bins=30, ax=axes[0], kde=True).set_title("Distribution of Mean User Ratings")
sns.histplot(user_features["active_years"], bins=30, ax=axes[1], kde=True).set_title("Distribution of User Activity Span (Years)")
plt.show()

## 3. Item Feature Engineering

We extract content-based features for movies:
1. **Genres**: Multi-hot encoded categorical variables.
2. **Tags**: TF-IDF vectors derived from user-assigned tags.
3. **Stats**: Popularity and average rating metrics.

In [ ]:
genre_features = build_genre_features(dataset["movies"])
tag_features = build_tag_features(dataset["tags"], dataset["movies"], max_features=100) # Small sample for demo
item_features = build_item_features(dataset["movies"], dataset["ratings"], genre_features, tag_features)

print(f"Item features shape: {item_features.shape}")
item_features.head()

In [ ]:
plt.figure(figsize=(10, 6))
item_features.filter(like="genre_").sum().sort_values(ascending=False).plot(kind="bar")
plt.title("Movie Count by Genre")
plt.ylabel("Number of Movies")
plt.show()

## 4. Interaction Matrix

Finally, we construct a compressed sparse row (CSR) matrix representing the interactions between users and movies. This is the primary input for most collaborative filtering algorithms.

In [ ]:
interactions, mappings = build_interaction_matrix(dataset["ratings"])
sparsity = 1.0 - (interactions.nnz / (interactions.shape[0] * interactions.shape[1]))

print(f"Matrix Shape: {interactions.shape} (Users x Movies)")
print(f"Non-zero entries: {interactions.nnz}")
print(f"Sparsity: {sparsity:.2%}")

## Summary of Features Exported

1. **`user_features.pkl`**: Numerical profile for each user.
2. **`item_features.pkl`**: Content-based and popularity features for each movie.
3. **`interaction_matrix.npz`**: The sparse rating matrix.
4. **`interaction_mappings.json`**: Mappings between internal matrix indices and original IDs.